# 05 — Fundamental & Quality Alpha Research

Fundamental alphas use accounting data (balance sheet, income statement, cash flow)
to find mispriced stocks. They tend to have **lower turnover** than price-based
alphas — fundamentals update quarterly — which benefits the fitness formula.

## Alpha ideas by theme

**Value**
- Book-to-market: `group_rank(book_to_price, subindustry)` — buy cheap within sector
- Earnings yield: `group_rank(earnings_per_share / close, subindustry)`

**Quality**
- ROE trend: `group_rank(ts_delta(roe, 4), subindustry)` — buy improving profitability
- Gross margin momentum: `ts_delta(gross_margin, 4)` — improving margins → outperformance
- Asset growth penalty: `-rank(ts_delta(total_assets, 4))` — firms that grow assets excessively underperform

**Profitability**
- Operating leverage: `rank(operating_income / revenue)` 
- Cash conversion: `rank(cash_from_operations / net_income)` — quality earnings vs paper earnings

## Key settings for fundamental alphas

- `decay=10–20` — fundamental signals update slowly; smooth the signal
- `neutralization='Subindustry'` — ALWAYS neutralize fundamentals within sector to avoid sector bets
- `delay=1` — fundamentals are reported with a delay; use delay=1 to avoid look-ahead bias

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from wq.client import get_client, simulate
from wq.alpha import AlphaConfig, fundamental_alphas, FUNDAMENTAL_TEMPLATES, expand_templates
from wq.utils import load_datafields, search_fields, load_results, top_alphas, parse_result

client = get_client()

## Step 1: Find relevant fundamental fields from the catalog

In [ ]:
catalog = load_datafields()

# Look for value-related fields
value_fields = search_fields(catalog, keyword='book', category='fundamental')
print(f'Book-related fields: {len(value_fields)}')
value_fields.head(10)

In [ ]:
# ROE / profitability fields
quality_fields = search_fields(catalog, keyword='return on equity')
quality_fields2 = search_fields(catalog, keyword='roe')
pd.concat([quality_fields, quality_fields2]).drop_duplicates(subset='id').head(15)

In [ ]:
# Margin-related fields
margin_fields = search_fields(catalog, keyword='margin')
print(f'Margin fields: {len(margin_fields)}')
margin_fields.head(10)

## Step 2: Define hand-crafted fundamental alphas

After exploring the catalog, we specify specific alphas based on economic intuition.

In [ ]:
# Replace field IDs below with actual IDs from your catalog search
# Use: search_fields(catalog, keyword='...').iloc[0]['id']

FUNDAMENTAL_SETTINGS = dict(
    universe='TOP3000',
    region='USA',
    neutralization='Subindustry',
    delay=1,
    decay=10,  # smooth fundamental signals
)

fundamental_configs = [
    # Value: buy cheap stocks within their subindustry
    AlphaConfig(
        expression="group_rank(book_to_price, subindustry)",
        tags=['fundamental', 'value'],
        notes="Book-to-price value factor, subindustry ranked",
        **FUNDAMENTAL_SETTINGS,
    ),
    # Quality: improving ROE
    AlphaConfig(
        expression="group_rank(ts_delta(roe, 4), subindustry)",
        tags=['fundamental', 'quality', 'roe'],
        notes="Stocks with improving ROE over last 4 quarters",
        **FUNDAMENTAL_SETTINGS,
    ),
    # Asset growth penalty
    AlphaConfig(
        expression="-1 * group_rank(ts_delta(total_assets, 4), subindustry)",
        tags=['fundamental', 'quality', 'accruals'],
        notes="Penalise aggressive asset growth (overinvestment)",
        **FUNDAMENTAL_SETTINGS,
    ),
    # Earnings quality: cash earnings vs accruals
    AlphaConfig(
        expression="group_rank(cash_from_operations / (net_income + 1e-6), subindustry)",
        tags=['fundamental', 'quality', 'accruals'],
        notes="Cash conversion quality — real earnings vs paper earnings",
        **FUNDAMENTAL_SETTINGS,
    ),
]

print(f'{len(fundamental_configs)} fundamental alphas defined')
for c in fundamental_configs:
    print(f'  {c.expression}')

## Step 3: Run a systematic sweep over fundamental fields

In [ ]:
# Identify the top-N most-used fundamental fields and auto-generate alphas
count_col = next((c for c in catalog.columns if 'alphacount' in c.lower()), None)
cat_col = next((c for c in catalog.columns if 'category' in c.lower() and 'sub' not in c.lower()), None)
id_col = next((c for c in catalog.columns if c.lower() == 'id'), None)

if cat_col and count_col and id_col:
    top_fundamental = (
        catalog[catalog[cat_col].str.lower().str.contains('fundamental', na=False)]
        .nlargest(10, count_col)[id_col].tolist()
    )
    print('Top fundamental fields (by alphaCount):', top_fundamental)

    sweep_configs = []
    for field_id in top_fundamental:
        for expr in expand_templates(FUNDAMENTAL_TEMPLATES, field_id, window=4):
            sweep_configs.append(AlphaConfig(
                expression=expr,
                tags=['fundamental', 'sweep', field_id],
                **FUNDAMENTAL_SETTINGS,
            ))
    print(f'{len(sweep_configs)} sweep alphas generated')
else:
    sweep_configs = []
    print('Could not identify fundamental fields — check catalog column names above')

In [ ]:
# Combine hand-crafted + sweep
all_fundamental = fundamental_configs + sweep_configs

# De-duplicate by fingerprint
existing = load_results()
done = set(existing['fingerprint'].dropna()) if not existing.empty else set()
to_run = [c for c in all_fundamental if c.fingerprint() not in done]
print(f'Running {len(to_run)} fundamental alphas')

In [ ]:
for config in tqdm(to_run, desc='Fundamental sweep'):
    try:
        result = simulate(client, config, save=True)
        m = parse_result(result)
        print(f'  sharpe={m["is_sharpe"]}  fitness={m["is_fitness"]}  {config.expression[:60]}')
    except Exception as e:
        print(f'  ERROR ({config.expression[:50]}): {e}')

## Results

In [ ]:
all_results = load_results()
fund_results = all_results[all_results['tags'].str.contains('fundamental', na=False)].copy()
fund_results = fund_results.dropna(subset=['is_sharpe'])

print(f'{len(fund_results)} fundamental results')
top_alphas(fund_results, n=15)[['expression', 'is_sharpe', 'is_fitness', 'is_returns', 'is_turnover', 'decay']]